In [ ]:
import os
import sys

sys.path.append("/home/justin/code/point-to-pose/")
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import glob
import copy
import open3d as o3d
from omegaconf import OmegaConf


import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Increase the embedding limit (set to 50 MB)
plt.rcParams["animation.embed_limit"] = 500

import torch
import torch.nn.functional as F

from point2pose.core.build import build_from_cfg
from point2pose.core.module_registry import TRACKER
from point2pose.modules.tracker.cotracker import CoTrackerRealtimeTracker
from point2pose.data_types.frame import Frame


In [ ]:
config_pth = "/home/justin/code/point-to-pose/configs/pipeline/pipeline_test2.yaml"
data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/edamame_box"

num_points = 5


In [ ]:

# Function to load RGB images
def load_rgb_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load RGB images from the /rgb subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.jpg', '.jpeg', '.png'])
    
    Returns:
        List[np.ndarray]: List of RGB images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.jpg', '.jpeg', '.png']
    
    rgb_folder = os.path.join(folder_path, 'rgb')
    if not os.path.exists(rgb_folder):
        raise FileNotFoundError(f"RGB folder not found: {rgb_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(rgb_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            img = cv2.imread(file_path)
            if img is not None:
                # Convert BGR to RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img_rgb)
    
    return images

# Function to load depth images
def load_depth_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load depth images from the /depth subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /depth subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.tiff', '.tif'])
    
    Returns:
        List[np.ndarray]: List of depth images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.png', '.tiff', '.tif']
    
    depth_folder = os.path.join(folder_path, 'depth')
    if not os.path.exists(depth_folder):
        raise FileNotFoundError(f"Depth folder not found: {depth_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(depth_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load depth image (usually 16-bit)
            img = cv2.imread(file_path, cv2.IMREAD_ANYDEPTH)
            if img is not None:
                images.append(img)
    
    return images

# Function to load mask images
def load_mask_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load mask images from the /masks subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /masks subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.jpg', '.jpeg'])
    
    Returns:
        List[np.ndarray]: List of mask images as numpy arrays (binary or grayscale)
    """
    if file_extensions is None:
        file_extensions = ['.png', '.jpg', '.jpeg']
    
    masks_folder = os.path.join(folder_path, 'masks')
    if not os.path.exists(masks_folder):
        raise FileNotFoundError(f"Masks folder not found: {masks_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(masks_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load mask as grayscale
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    
    return images

# Function to load all image types together
def load_all_images(folder_path: str) -> Dict[str, List[np.ndarray]]:
    """
    Load RGB, depth, and mask images from the specified folder structure.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb, /depth, and /masks subdirectories
    
    Returns:
        Dict[str, List[np.ndarray]]: Dictionary with keys 'rgb', 'depth', 'masks' containing lists of images
    """
    result = {}
    
    try:
        result['rgb'] = load_rgb_images(folder_path)
        print(f"Loaded {len(result['rgb'])} RGB images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['rgb'] = []
    
    try:
        result['depth'] = load_depth_images(folder_path)
        print(f"Loaded {len(result['depth'])} depth images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['depth'] = []
    
    try:
        result['masks'] = load_mask_images(folder_path)
        print(f"Loaded {len(result['masks'])} mask images")
    except FileNotFoundError as e:
        print(f"Warning: {e}")
        result['masks'] = []
    
    return result

# Function to get file paths without loading images
def get_image_paths(folder_path: str) -> Dict[str, List[str]]:
    """
    Get file paths for RGB, depth, and mask images without loading them.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb, /depth, and /masks subdirectories
    
    Returns:
        Dict[str, List[str]]: Dictionary with keys 'rgb', 'depth', 'masks' containing lists of file paths
    """
    result = {}
    
    # RGB paths
    rgb_folder = os.path.join(folder_path, 'rgb')
    if os.path.exists(rgb_folder):
        rgb_files = []
        for ext in ['.jpg', '.jpeg', '.png']:
            pattern = os.path.join(rgb_folder, f"*{ext}")
            rgb_files.extend(glob.glob(pattern))
        result['rgb'] = sorted(rgb_files)
    else:
        result['rgb'] = []
    
    # Depth paths
    depth_folder = os.path.join(folder_path, 'depth')
    if os.path.exists(depth_folder):
        depth_files = []
        for ext in ['.png', '.tiff', '.tif']:
            pattern = os.path.join(depth_folder, f"*{ext}")
            depth_files.extend(glob.glob(pattern))
        result['depth'] = sorted(depth_files)
    else:
        result['depth'] = []
    
    # Mask paths
    masks_folder = os.path.join(folder_path, 'masks')
    if os.path.exists(masks_folder):
        mask_files = []
        for ext in ['.png', '.jpg', '.jpeg']:
            pattern = os.path.join(masks_folder, f"*{ext}")
            mask_files.extend(glob.glob(pattern))
        result['masks'] = sorted(mask_files)
    else:
        result['masks'] = []
    
    return result

def wait_for_click_and_get_pixel(image):
    """
    Display an image in a Jupyter notebook and wait for a mouse click.
    Returns (x, y) pixel coordinates of the click.
    """
    coords = []

    def onclick(event):
        # Only respond to left mouse button clicks
        if event.button == 1 and event.xdata is not None and event.ydata is not None:
            coords.append((int(event.xdata), int(event.ydata)))
            plt.close()  # Close the figure after click

    fig, ax = plt.subplots()
    ax.imshow(image)
    ax.set_title("Click on the image")
    cid = fig.canvas.mpl_connect('button_press_event', onclick)

    plt.show()

    if coords:
        return coords[0]
    else:
        return None

def load_camera_param(folder_path: str) -> Dict[str, np.ndarray]:
    """
    Load camera parameters from the specified folder.
    
    Args:
        folder_path (str): Path to the main folder containing /camera_param subdirectory
    
    Returns:
        Dict[str, np.ndarray]: Dictionary with keys 'K', 'D' containing camera parameters
    """
    # load txt
    cam_param_path = os.path.join(folder_path, 'cam_K.txt')
    # if not os.path.exists(camera_param_folder):
    #     raise FileNotFoundError(f"Camera parameter folder not found: {camera_param_folder}")
    
    # load the camera parameters from the camera_param subdirectory
    # with open(cam_param_path, 'r') as file:
    #     lines = [line.strip() for line in file]
    return np.loadtxt(cam_param_path)


In [ ]:
### simple point sampling function ###
def sample_random_points_in_mask(mask, num_points=10, min_distance=5):
    """
    Sample random points within a binary segmentation mask.
    
    Args:
        mask: Binary mask (H, W) where 1 indicates foreground
        num_points: Number of points to sample
        min_distance: Minimum distance between sampled points (pixels)
    
    Returns:
        points: Array of shape (num_points, 2) with (x, y) coordinates
    """
    import numpy as np
    
    # Find all valid pixel coordinates within the mask
    y_coords, x_coords = np.where(mask > 0)
    
    if len(y_coords) == 0:
        print("Warning: No valid pixels found in mask")
        return np.array([]).reshape(0, 2)
    
    # Convert to (x, y) format
    valid_pixels = np.stack([x_coords, y_coords], axis=1)
    
    if len(valid_pixels) < num_points:
        print(f"Warning: Only {len(valid_pixels)} valid pixels available, returning all")
        return valid_pixels
    
    # Sample points with minimum distance constraint
    sampled_points = []
    attempts = 0
    max_attempts = num_points * 100  # Prevent infinite loops
    
    while len(sampled_points) < num_points and attempts < max_attempts:
        # Randomly select a pixel
        idx = np.random.randint(0, len(valid_pixels))
        candidate = valid_pixels[idx]
        
        # Check if it's far enough from already sampled points
        if len(sampled_points) == 0:
            sampled_points.append(candidate)
        else:
            distances = [np.linalg.norm(candidate - pt) for pt in sampled_points]
            if min(distances) >= min_distance:
                sampled_points.append(candidate)
        
        attempts += 1
    
    if len(sampled_points) < num_points:
        print(f"Warning: Could only sample {len(sampled_points)} points with distance constraint")
    
    return np.array(sampled_points)


In [ ]:
### plotting related functions ###

def draw_points_on_image(image, points, colors):
    # if points are tensor
    # print(points.shape)
    if isinstance(points, torch.Tensor):
        points = points.cpu().numpy()
        
    for i in range(points.shape[0]):
        cv2.circle(
            image,
            points[i, :].astype(int).reshape(2),
            radius=5,
            color=colors[i],
            thickness=-1,
        )

def get_n_colors(n):
        cmap = plt.get_cmap("RdYlGn")  # or 'tab20', 'jet', etc.
        colors = [tuple(int(c * 255) for c in cmap(i / n)[:3]) for i in range(n)]
        return colors

def get_n_uncertainty_colors(uncertainties, u_min=0.0, u_max=1.0, inverse=False):
        cmap = plt.get_cmap("jet")  # or 'tab20', 'jet', etc.
        norm_uncertainties = (uncertainties - u_min) / (u_max - u_min + 1e-8)
        if inverse:
            norm_uncertainties = 1 - norm_uncertainties
        colors = [tuple(int(c * 255) for c in cmap(u)[:3]) for u in norm_uncertainties]
        return colors
        
def visualize_out_frames_slider(out_frames):
    # Create slider
    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(out_frames)-1,
        step=1,
        description='Frame:',
        continuous_update=False
    )

    # Create output widget
    output = widgets.Output()

    def on_value_change(change):
        with output:
            output.clear_output(wait=True)
            plt.figure(figsize=(10, 8))
            plt.imshow(out_frames[change['new']])
            plt.title(f"Frame {change['new']}")
            plt.axis('off')
            plt.show()

    slider.observe(on_value_change, names='value')

    # Display widgets
    display(slider, output)
    # Trigger initial display
    on_value_change({'new': 0})




def visualize_out_frames_animation(out_frames):

    # Your existing animation code
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_title("Image Sequence")

    def animate(frame):
        ax.clear()
        ax.imshow(out_frames[frame])
        ax.set_title(f"Frame {frame}")
        ax.axis('off')
        return ax,

    anim = animation.FuncAnimation(
        fig, animate, frames=len(out_frames), 
        interval=100, repeat=True, blit=False
    )

    display(HTML(anim.to_jshtml()))
    plt.close(fig)

    return anim


In [ ]:
### load data ###
data = load_all_images(data_path)
data['K'] = load_camera_param(data_path)


In [ ]:
### Initialize CoTracker tracker ###

cotracker_cfg = OmegaConf.load(config_pth)
# If config has nested structure, extract tracker part
if 'tracker' in cotracker_cfg:
    cotracker_cfg = cotracker_cfg.tracker

tracker = build_from_cfg(cotracker_cfg, TRACKER)


In [ ]:
# initialize the first image 
rgb_image_first = data["rgb"][0]
depth_image_first = data["depth"][0]
mask = data["masks"][0]

# Create Frame object for first frame
frame_first = Frame(
    id=0,
    rgb=rgb_image_first,
    depth=depth_image_first,
    intrinsics=data['K']
)

sampled_points = sample_random_points_in_mask(mask, num_points=num_points, min_distance=5)
point_colors = get_n_colors(len(sampled_points))

# visualize sampled points with colored mask
rgb_image_first_vis2 = rgb_image_first.copy()
draw_points_on_image(rgb_image_first_vis2, sampled_points, point_colors)
plt.imshow(rgb_image_first_vis2)
plt.show()


In [ ]:
# add initial points to tracker
tracker.add_query_points(frame_first, sampled_points)


In [ ]:
# loop through the rest of the images
# repeat first rgb image and put them in the first 15 images in data["rgb"]

rgb_data = data["rgb"][0]
data["rgb"] = [rgb_data for _ in range(15)] + data["rgb"][1:]


out_frames = [rgb_image_first_vis2]
for i in range(len(data["rgb"])):

    rgb_image = data["rgb"][i]
    
    # Create Frame object
    frame = Frame(
        id=i,
        rgb=rgb_image,
        depth=data["depth"][i] if i < len(data["depth"]) else None,
        intrinsics=data['K']
    )
    
    tracks, uncertainty, visibles = tracker.track_once(frame)

    if tracks is None or len(tracks) == 0:
        continue
    
    # Visualize the current frame with tracking information
    rgb_image_vis = rgb_image.copy()
    # tracks_xy = tracks[:, [1, 0]]  # swap y and x
    draw_points_on_image(rgb_image_vis, tracks, get_n_colors(len(tracks)))
    out_frames.append(rgb_image_vis)


In [ ]:
visualize_out_frames_slider(out_frames)


In [ ]:
anim = visualize_out_frames_animation(out_frames)
